# STAT 764 · Meeting 12 — Calibration lab

**Thursday, October 15**

On Tuesday we learned to *check* a model's numbers: ten groups, what the model said against what
happened. Today is what to do when the check fails.

Last Thursday's notebook (Meeting 10, Section 3) built a model with the most common advice for
imbalanced data — *reweight the classes*, so each readmission counts about eight times as much
while the model is fitted. It ranks patients exactly as well as ours, and its numbers are about
four times too high: it says 0.47 on average when 0.11 come back. Lab 5 asked what that does to
the Brier score. Today we fix it **without refitting the model**.

**The question for today.** The order is good and the numbers are wrong. Can we fix the numbers
without touching the order — and on which patients do we learn the fix?

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logit
from stat764 import diabetes

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# The readmission model from Meetings 9-11, rebuilt so this notebook stands on its own.
enc = diabetes()
enc = enc[~enc["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]  # death, hospice
y_all = (enc["readmitted"] == "<30").astype(int)

NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
X_all = enc[NUM + CAT].astype({c: str for c in CAT})

# Split by PATIENT, so nobody is on both sides -- the same split as Meetings 9-11.
train, test = next(GroupShuffleSplit(1, test_size=0.25, random_state=764)
                   .split(X_all, y_all, groups=enc["patient_nbr"]))


def readmission_model(**options):
    """The Meeting 9 pipeline. `options` go to LogisticRegression (e.g. class_weight=...)."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), NUM),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
        ])),
        ("model", LogisticRegression(max_iter=2000, **options)),
    ])


# Two models, both fitted on the same training patients:
model = readmission_model().fit(X_all.iloc[train], y_all.iloc[train])
reweighted = readmission_model(class_weight="balanced").fit(X_all.iloc[train],
                                                            y_all.iloc[train])
p = model.predict_proba(X_all.iloc[test])[:, 1]           # our model's probabilities
p_rw = reweighted.predict_proba(X_all.iloc[test])[:, 1]   # the reweighted model's
y = y_all.iloc[test].to_numpy()                           # 1 = back within 30 days

print(f"  {len(y):,} test discharges · readmitted within 30 days: {y.mean():.1%}")
print(f"  our model:  AUC {roc_auc_score(y, p):.3f} · average said {p.mean():.3f}")
print(f"  reweighted: AUC {roc_auc_score(y, p_rw):.3f} · average said {p_rw.mean():.3f}")

Tuesday's helpers, unchanged — plus `log_odds`, the scale logistic regression works on. Nothing
new to learn in this cell.

In [ ]:
READMISSION = 17_700           # dollars -- AHRQ HCUP Statistical Brief #307 (2024), 2020 data
P_STAR = 150 / (0.05 * READMISSION)    # Meeting 10's break-even for a $150 visit: 0.169


def net_per_1000(probs, outcome, threshold=P_STAR, cost=150, prevents=0.05):
    """Meeting 10's money: visit everyone at or above `threshold`. Saved minus spent, per 1,000."""
    visit = probs >= threshold
    saved = (visit & (outcome == 1)).sum() * prevents * READMISSION
    spent = visit.sum() * cost
    return (saved - spent) / len(outcome) * 1000


def by_decile(probs, outcome):
    """Ten equal groups by probability: what the model SAID against what HAPPENED in each."""
    group = pd.qcut(probs, 10, labels=False, duplicates="drop")
    return pd.DataFrame({"said": [probs[group == k].mean() for k in np.unique(group)],
                         "came back": [outcome[group == k].mean() for k in np.unique(group)]},
                        index=pd.Index(np.unique(group) + 1, name="tenth")).round(3)


def reliability_plot(versions, outcome, ax=None, title=""):
    """One point per tenth of patients: x = what the model said, y = what happened.
    A calibrated model's points sit on the dashed diagonal."""
    ax = ax or plt.subplots(figsize=(4.6, 4.6))[1]
    top = 0.0
    for name, probs in versions.items():
        table = by_decile(probs, outcome)
        ax.plot(table["said"], table["came back"], marker="o", label=name)
        top = max(top, table.to_numpy().max())
    top = min(1.0, top * 1.1)
    ax.plot([0, top], [0, top], ls="--", color="gray", label="calibrated")
    ax.set(xlim=(0, top), ylim=(0, top), xlabel="the model said (average in the tenth)",
           ylabel="actually came back (share)", title=title)
    ax.legend(loc="upper left", fontsize=8)
    return ax


def log_odds(probs):
    """log(p / (1 - p)), kept finite at 0 and 1."""
    return logit(np.clip(probs, 1e-6, 1 - 1e-6))


def calibration_slope(probs, outcome):
    """Tuesday's slope: 1 = the spread is right; below 1 = too extreme; above 1 = too timid."""
    refit = LogisticRegression(C=1e6, max_iter=1000).fit(log_odds(probs).reshape(-1, 1), outcome)
    return refit.coef_[0, 0]

### Two halves of the test patients

A fix has to be **learned** on patients whose outcomes we know, and **judged** on patients it has
never seen — otherwise we would be grading it on its own homework. So split the test patients in
two, by patient: **`cal`** (learn every fix here) and **`check`** (judge everything here). Neither
model was fitted on any of them.

Every number from here on is on the `check` half, so it differs a little from the full test
set's — about half the patients, chosen at random.

In [ ]:
# Split the TEST patients in two, by patient: cal = learn the fix; check = judge it.
test_patients = enc["patient_nbr"].iloc[test].to_numpy()
everyone = np.unique(test_patients)
cal_patients = np.random.default_rng(12).choice(everyone, len(everyone) // 2, replace=False)
cal = np.isin(test_patients, cal_patients)     # True/False for each test discharge
check = ~cal

print(f"  cal:   {cal.sum():,} discharges · readmitted {y[cal].mean():.1%}   (learn fixes here)")
print(f"  check: {check.sum():,} discharges · readmitted {y[check].mean():.1%}   (judge everything here)")

## 1. One number for the order and the numbers: the Brier score

The **Brier score** is the average of (probability − what happened)², where what happened is 1
for a patient who came back and 0 for one who did not. **Lower is better.** A patient given 0.2
who came back adds (0.2 − 1)² = 0.64; one given 0.2 who did not adds (0.2 − 0)² = 0.04. Lab 5
introduced it as `brier_score_loss(outcome, probs)`.

It punishes both kinds of mistake at once: putting patients in the wrong order, *and* giving them
the wrong numbers. The yardstick is the **base rate**: give every patient the same number — the
share who came back in training — which knows nothing about anyone. A model worth having should
beat it.

In [ ]:
# Three forecasts for the same check patients. The base rate gives everyone one number.
base_rate = y_all.iloc[train].mean()          # the training patients' readmission rate
forecasts = {"our model": p[check], "reweighted": p_rw[check],
             "base rate, for everyone": np.full(check.sum(), base_rate)}

print(f"  {'':26}{'AUC':>7}{'average said':>14}{'Brier':>9}")
for name, probs in forecasts.items():
    print(f"  {name:26}{roc_auc_score(y[check], probs):>7.3f}{probs.mean():>14.3f}"
          f"{brier_score_loss(y[check], probs):>9.4f}")
print(f"  {'what happened':26}{'':>7}{y[check].mean():>14.3f}")

**What this shows.** The reweighted model ranks the check patients as well as ours — AUC 0.669
against 0.668 — and its Brier score is more than twice the base rate's: 0.2254 against 0.0980.
A forecast that knows nothing about any patient beats it, because its numbers are about four
times too high for everyone. Our model beats the base rate, by a little: 0.0940.

That is Lab 5's question (a), answered: **AUC sees only the order; the Brier score sees the order
and the numbers**, and wrong numbers cost more than a good order earns.

## 2. Fix the numbers, keep the order

If the order is good and the numbers are wrong, learn a map from what the model **said** to what
**came back**, on patients whose outcomes you know — then apply it to new patients. This is
**recalibration**. Two standard maps:

- **Platt scaling** is Tuesday's calibration-slope fit, put to work. Fit a logistic regression of
  what happened on the model's own log-odds; it learns a **slope** and an **intercept**. Then
  push new patients' log-odds through that same line. One straight line on the log-odds can
  stretch, squeeze or shift the numbers, but it **never changes the order** — so AUC cannot move.
- **Isotonic regression** learns any never-decreasing step from what the model said to what
  happened. More flexible; it can tie patients together. The studio tries it.

Learned on the `cal` patients; judged on `check`.

In [ ]:
# Platt scaling, learned on the CAL patients: what happened ~ the reweighted model's log-odds.
platt = LogisticRegression(C=1e6, max_iter=1000)               # C=1e6: effectively no penalty
platt.fit(log_odds(p_rw[cal]).reshape(-1, 1), y[cal])
print(f"  learned on {cal.sum():,} cal discharges: slope {platt.coef_[0, 0]:.2f},"
      f" intercept {platt.intercept_[0]:.2f}\n")

# Apply the learned line to the CHECK patients, whom the fix never saw.
p_fixed = platt.predict_proba(log_odds(p_rw[check]).reshape(-1, 1))[:, 1]

versions = {"our model": p[check], "reweighted": p_rw[check], "reweighted, fixed": p_fixed}
print(f"  {'':20}{'AUC':>7}{'average':>9}{'slope':>7}{'Brier':>9}{'visits':>8}{'net per 1,000':>15}")
for name, probs in versions.items():
    print(f"  {name:20}{roc_auc_score(y[check], probs):>7.3f}{probs.mean():>9.3f}"
          f"{calibration_slope(probs, y[check]):>7.2f}{brier_score_loss(y[check], probs):>9.4f}"
          f"{np.mean(probs >= P_STAR):>8.1%}{net_per_1000(probs, y[check]):>15,.0f}")
print(f"  {'what happened':20}{'':>7}{y[check].mean():>9.3f}")

reliability_plot(versions, y[check], title="The reweighted model, before and after Platt")
plt.tight_layout()
plt.show()

**What this shows.** Same AUC before and after, 0.669 — the fix never touched the order. Every
other column moved. The average went from 0.466 to 0.112, where 0.110 came back; the Brier score
from 0.2254 to 0.0940, the same as our model's; and Meeting 10's \$150 rule went from visiting
99.6% of patients and losing \$52,142 per 1,000 to visiting 10.7% and earning \$6,744.

The learned line says what was wrong: an intercept of −2.06 pulls every patient's log-odds down a
long way (the numbers were far too **high** everywhere), and a slope of 0.81 squeezes them a little
(slightly too **extreme**). On the diagram the fixed model sits on the diagonal beside ours.

### Platt scaling, step by step

1. **Put the scores on the log-odds scale**, log(p / (1 − p)). Probabilities are trapped between 0
   and 1; log-odds run along the whole number line, with 0.5 at zero:

   | probability | 0.10 | 0.20 | 0.50 | 0.80 | 0.90 |
   |---|---|---|---|---|---|
   | log-odds | −2.20 | −1.39 | 0 | +1.39 | +2.20 |

   **Why log-odds?** Two reasons. A straight line drawn on *probabilities* eventually runs below 0
   or above 1 — it would give some patients a negative chance. And on the probability scale the
   same change in risk looks different in different places: doubling a patient's odds moves 0.01 to
   about 0.02 but 0.50 to 0.67. On the log-odds scale every doubling of the odds is the same step,
   +0.69, anywhere on the line — so "what the model said" against "what happened" tends to be a
   straight line there, and a straight line is what logistic regression fits.

2. **Fit a straight line** on the `cal` patients: log-odds of really coming back = **a + b ×** the
   model's log-odds. It is an ordinary logistic regression with one input — the model's own score.
3. **Read the two numbers.** An honest model would give a = 0 and b = 1 (change nothing). *a* is
   the true log-odds for a patient the model called 0.5 (its log-odds are 0), so a = −2.06 says
   those patients really come back about 11% of the time. *b* below 1 means the model exaggerates
   the differences between patients; above 1, it understates them.
4. **Push every new score through the line** and turn the result back into a probability.

The picture below draws the same fit twice: as a straight line on the log-odds scale, and as the
curve it becomes in ordinary probabilities. (It is a sigmoid of the model's *log-odds*, not of the
probability itself, so on a 0-to-1 axis it bends rather than looking like the textbook S.) The dots are
twenty groups of `cal` patients.

In [ ]:
from scipy.special import expit       # expit turns log-odds back into a probability

a, b = platt.intercept_[0], platt.coef_[0, 0]    # the learned line

# Twenty equal groups of CAL patients by score: what the model said, and what really happened.
group = pd.qcut(p_rw[cal], 20, labels=False)
said20 = np.array([p_rw[cal][group == k].mean() for k in range(20)])
share20 = np.array([y[cal][group == k].mean() for k in range(20)])

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 5.4))
examples = ((0.2, "C2"), (0.5, "C3"), (0.8, "C4"))

# Left: the log-odds scale, where Platt is a straight line through the groups.
xs = np.linspace(-3.5, 3.5, 100)
left.plot(xs, xs, ls="--", color="gray", label="no change: the model's own numbers")
left.plot(xs, a + b * xs, lw=2.5, label=f"Platt's line: {a:.2f} + {b:.2f} × model's log-odds")
left.scatter(log_odds(said20), log_odds(share20), color="C1", zorder=3,
             label="cal patients, 20 groups: what really happened")
for s, c in examples:                       # each patient moves straight down onto the line
    lx = log_odds(np.array([s]))[0]
    left.annotate("", xy=(lx, a + b * lx), xytext=(lx, lx), arrowprops=dict(arrowstyle="->", color=c, lw=2))
    left.text(lx + 0.08, (lx + a + b * lx) / 2, f"said {s}", color=c, fontsize=9,
              bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
left.set(xlabel="the model's log-odds", ylabel="log-odds of really coming back",
         xlim=(-3.5, 3.5), ylim=(-4.5, 3.5), title="On the log-odds scale, Platt is a straight line")
left.legend(loc="upper left", fontsize=8.5)

# Right: the same line drawn in probabilities becomes a curve -- one that only goes up and never
# leaves 0 to 1.
ps = np.linspace(0.005, 0.995, 300)
right.plot([0, 1], [0, 1], ls="--", color="gray", label="no change")
right.plot(ps, expit(a + b * log_odds(ps)), lw=2.5, label="Platt's map, drawn in probabilities")
right.scatter(said20, share20, color="C1", zorder=3, label="cal patients, 20 groups")
for s, c in examples:
    fixed = expit(a + b * log_odds(np.array([s]))[0])
    right.annotate("", xy=(s, fixed), xytext=(s, s), arrowprops=dict(arrowstyle="->", color=c, lw=2))
    right.text(s + 0.02, fixed + 0.06, f"{s} → {fixed:.2f}", color=c, fontsize=10, fontweight="bold",
               bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
right.set(xlabel="the model said", ylabel="the fixed probability", xlim=(0, 1), ylim=(0, 1),
          title="In probabilities, the same line becomes a curve")
right.legend(loc="upper left", fontsize=8.5)
plt.tight_layout()
plt.show()

**What this shows.** On the left, the orange dots sit well **below** the grey line: for every
group, the model said more than happened. Platt's blue line runs through the dots — lower than the
grey line by about 2 at the centre (that is *a*) and a little flatter (that is *b*, 0.81). Each
arrow is one score pushed straight down from what the model said onto the line. On the right, the
same three arrows land at 0.04, 0.11 and 0.28, and the curve **only goes up**, so a higher score
always keeps a higher probability: the order, and AUC, cannot change.

### Isotonic regression, step by step

Isotonic regression makes no assumption about shape. It only insists that the map **never goes
down** — a higher score may not get a lower probability. Using the same twenty groups:

1. **Write down each group's share who came back**, lowest scores first.
2. **Find a dip**: a group whose share is *lower* than the group just below it.
3. **Pool the two**: give both groups their combined share (all who came back ÷ all patients in
   the two). That is called *pooling adjacent violators*.
4. **Repeat** until the shares only go up. What is left is a staircase — the isotonic map.

The real fit does exactly this patient by patient, on all 12,422 `cal` discharges; twenty groups
make it visible.

In [ ]:
from sklearn.isotonic import IsotonicRegression

# The twenty groups' raw shares, and where they dip below the group before.
size20 = np.bincount(group)                                   # patients in each group
dips = [k for k in range(1, 20) if share20[k] < share20[k - 1]]
pooled = IsotonicRegression().fit(said20, share20, sample_weight=size20).predict(said20)
print(f"  {len(dips)} of the 20 groups dip below the group before them;"
      f" pooling leaves {len(np.unique(pooled.round(6)))} steps")

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.plot(said20, share20, ":", color="C1", lw=1)
ax.scatter(said20, share20, color="C1", zorder=3, label="each group's share who came back")
ax.scatter(said20[dips], share20[dips], s=120, facecolors="none", edgecolors="red", lw=1.8,
           zorder=4, label="a dip: lower than the group before")
ax.step(said20, pooled, where="mid", lw=2.5, color="C0", label="the isotonic staircase (dips pooled)")
ax.set(xlabel="the reweighted model said (group average)", ylabel="share who came back",
       title="Isotonic regression: pool every dip until the steps only go up")
ax.legend(loc="upper left", fontsize=8.5)
plt.tight_layout()
plt.show()

**What this shows.** The orange dots wander: seven of the twenty dip below the group before them —
noise, not a real drop in risk. Isotonic regression pools each dip with its neighbour until the
steps only go up, leaving thirteen steps. Unlike Platt it can follow any shape the data show, but
each step is only as trustworthy as the patients under it.

### What the two maps look like

A recalibration map is just a function: in goes what the model said, out comes the fixed
probability. Here are both maps, learned on the same `cal` patients from the reweighted model —
Platt's smooth curve and isotonic regression's staircase — with three patients walked through.
Platt works on log-odds: **z = intercept + slope × the model's log-odds**, then z goes back to a
probability. Isotonic looks the score up on its staircase.

In [ ]:
from sklearn.isotonic import IsotonicRegression

# Isotonic regression on the same CAL patients: the best never-decreasing staircase from what
# the reweighted model said to what happened. (Named `staircase` so the studio's names stay free.)
staircase = IsotonicRegression(out_of_bounds="clip").fit(p_rw[cal], y[cal])

# Three patients walked through each map.
said = np.array([0.20, 0.50, 0.80])
z = platt.intercept_[0] + platt.coef_[0, 0] * log_odds(said)     # Platt's line, on log-odds
by_platt = platt.predict_proba(log_odds(said).reshape(-1, 1))[:, 1]  # z back to a probability
by_staircase = staircase.predict(said)
print(f"  {'model said':>11}{'log-odds':>10}{'Platt z':>9}{'Platt':>8}{'isotonic':>10}")
for row in zip(said, log_odds(said), z, by_platt, by_staircase):
    print(f"  {row[0]:>11.2f}{row[1]:>10.2f}{row[2]:>9.2f}{row[3]:>8.2f}{row[4]:>10.2f}")

# Both maps across every score the reweighted model can give. The dashed line is "no fix".
grid = np.linspace(0.01, 0.99, 400)
curve = platt.predict_proba(log_odds(grid).reshape(-1, 1))[:, 1]
steps = staircase.predict(grid)
fig, ax = plt.subplots(figsize=(6.2, 4.4))
ax.plot(grid, curve, label="Platt: a smooth curve (two numbers)")
ax.step(grid, steps, where="post", label="isotonic: a staircase (follows the data)")
ax.plot([0, 1], [0, 1], ls="--", color="gray", label="no fix: fixed = said")
ax.set(xlabel="the reweighted model said", ylabel="the fixed probability",
       ylim=(0, 1.15 * max(curve.max(), steps.max())), title="Two maps, learned on the cal patients")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

**What this shows.** Both maps only go **up** — a higher score never gets a lower fixed
probability — so neither can change the order, and AUC stays at 0.669. Both pull the reweighted
model's numbers far down: a patient it called 0.50 becomes 0.11 either way, and one it called 0.80
becomes 0.28 (Platt) or 0.27 (isotonic).

The difference is the shape. **Platt** is one smooth curve set by two numbers: it cannot bend to
follow the data, but a handful of patients cannot fool it either. **Isotonic** follows the data step
by step — flexible, and only as good as the patients under each step. At the very top it jumps to
1.00: that step was learned from **one** patient, who happened to come back.

## 3. The rule: learn the fix on patients the model was not fitted on

A fix maps what the model *says* to what *happens*. So it has to be learned where the model says
what it will say in use — on patients it has never seen.

On its own training patients a model looks better than it is. An overfit model has partly
memorized them, so there its probabilities line up with what happened almost perfectly — and a
fix learned there concludes there is **nothing to fix**. On new patients the same model is too
extreme, and the fix you learned does not help. **Lab 6, Part 2** lets you watch this happen.

The second half of the rule is Meeting 8's: **the number you choose by is not the number to
report.** Learn the fix on one set of patients (`cal`), judge it on another (`check`). In practice
that third set comes from the training data — a calibration split, or cross-validation, which is
what scikit-learn's `CalibratedClassifierCV` does.

## Studio

Two questions to finish, the third if you have time. The tools:

- **learn a fix on `cal`, apply it to `check`** — Platt as in Section 2, or isotonic:

```python
from sklearn.isotonic import IsotonicRegression
iso = IsotonicRegression(out_of_bounds="clip").fit(probs[cal], y[cal])   # learn on cal
fixed = iso.predict(probs[check])                                        # apply to check
```

- **judge it on `check`**: the order, `roc_auc_score(y[check], fixed)`; the numbers,
  `by_decile(fixed, y[check])` and `brier_score_loss(y[check], fixed)`; the \$150 decision,
  `np.mean(fixed >= P_STAR)` and `net_per_1000(fixed, y[check])`.

`probs[cal]` and `y[cal]` always go together, and `probs[check]` with `y[check]` — the outcome for
the same patients.

**Before each question, write your guess** as a comment in its cell. Keep it when the answer
disagrees — the Compare uses it.

⏱ **During studio I will call each Mini-project 1 team over for its five-minute check-in.** Keep
working until it is your team's turn.

The notebook is due 24 hours after class. Done early? Lab 6 — Parts 2 and 3 use exactly today's
tools.

### 1. Turn a percentile into a probability

Tuesday's risk percentile, `rankdata(p) / len(p)`, ranks patients exactly as our model does, but
its numbers are not probabilities, so Meeting 10's \$150 rule visited 83% of patients and lost
money. Can a fix turn it into probabilities?

- **(a)** Make it again: `from scipy.stats import rankdata`, then `pct = rankdata(p) / len(p)`.
- **(b)** Learn isotonic regression on the `cal` patients — `pct[cal]` with `y[cal]` — and apply
  it to `pct[check]`. Call the result `pct_fixed`.
- **(c)** Judge three forecasts on `check`: `pct[check]` as it is, `pct_fixed`, and our model's
  `p[check]`. For each: AUC, average, the \$150 rule's visits and net per 1,000.

**Answer in a sentence:** did the fix make the percentile usable for the \$150 rule — and what did
it need that the percentile alone did not have?

### 2. Isotonic or Platt?

Section 2 fixed the reweighted model with Platt scaling (`p_fixed`).

- **(a)** Fix it again with isotonic regression, learned on `p_rw[cal]` with `y[cal]` and applied to
  `p_rw[check]`. Call the result `rw_iso`.
- **(b)** Compare `p_fixed` and `rw_iso` on `check`: AUC, average, Brier score, the \$150 rule's
  visits and net per 1,000 — and how many **different** probabilities each one gives,
  `len(np.unique(rw_iso))`.

**Answer in a sentence:** which would you use, and why? (Isotonic's flexibility has a price —
look at the last number.)

### 3. How many patients does a fix need? *(if you have time)*

Section 2 learned Platt on all 12,422 `cal` discharges. Learn it on far fewer, many times over:

- **(a)** `cal_rows = np.where(cal)[0]`. For each `s` in `range(20)`, draw
  `few = np.random.default_rng(s).choice(cal_rows, 200, replace=False)`, fit Platt on
  `log_odds(p_rw[few])` with `y[few]` (as in Section 2), and keep the slope, `.coef_[0, 0]`.
- **(b)** The smallest, middle and largest of the twenty slopes — `np.min`, `np.median`, `np.max`.
  Then the same with 1,000 discharges instead of 200.

**Answer in a sentence:** how far can a fix learned on 200 discharges land from the 0.81 learned
on all of `cal` — and what would a **negative** slope do to the order of the patients?

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

## Compare — at the start of next class

We compare answers at the start of next class, built from the notebooks you upload. So leave
these in your notebook, in words or as printed numbers:

1. Your guesses, written before you computed, next to what you found.
2. Question 1: the percentile's \$150-rule visits and net before and after the fix.
3. Question 2: which fix you would use, and the number that decided it.

## Exit ticket

> You recalibrate a model and its AUC does not change. In two or three sentences: why not — and
> what did change?

**Answer in the cell below, then save this notebook and upload it to Canvas under "Studio
notebook — Day 12". Your two studio points need this answered: a blank exit ticket earns zero.
It is not graded for being right — it is how I find out what landed.**

### Your exit ticket

*(double-click to edit, and replace this line)*

## Also in this neighborhood

📗 **Niculescu-Mizil & Caruana (2005), "Predicting good probabilities with supervised
learning,"** *ICML* — the classic comparison of Platt scaling and isotonic regression across
many kinds of model, and when each wins.

📗 **Landsgesell, Knoll & Wenzel (2026), arXiv 2603.08206** — today's optional reading: how
well the newest tabular foundation models' probabilities hold up under proper scoring rules, the
Brier score among them.

📗 **`CalibratedClassifierCV`** — scikit-learn's way to learn the fix inside cross-validation, so
no patient is used both to fit the model and to learn its fix.